In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Load COMPAS dataset ===
df = pd.read_csv("compas_cleaned.csv")

# === Extract features and label ===
X = df.drop(columns='two_year_recid')
y = df['two_year_recid']

# === Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Save group membership for fairness analysis
group_priv = X_test['race_African-American'] == 1   # privileged group
group_unpriv = X_test['race_African-American'] == 0 # non-privileged group

# === Define models ===
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Print header ===
print("\nFairness Metrics (Privileged: race_African-American == 1)")
print("--------------------------------------------------------------------------")
print(f"{'Model':<20} {'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")

# === Evaluate each model ===
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # === ERD ===
    err_priv = np.mean(y_pred[group_priv] != y_test[group_priv])
    err_unpriv = np.mean(y_pred[group_unpriv] != y_test[group_unpriv])
    erd = err_unpriv - err_priv

    # === TPRD ===
    def tpr(y_true, y_pred):
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_priv = tpr(y_test[group_priv], y_pred[group_priv])
    tpr_unpriv = tpr(y_test[group_unpriv], y_pred[group_unpriv])
    tprd = tpr_unpriv - tpr_priv

    # === ABROCA ===
    def compute_roc_auc(y_true, y_score):
        fpr, tpr_vals, _ = roc_curve(y_true, y_score)
        return auc(fpr, tpr_vals)

    abroca = np.nan
    if y_prob is not None:
        auc_priv = compute_roc_auc(y_test[group_priv], y_prob[group_priv])
        auc_unpriv = compute_roc_auc(y_test[group_unpriv], y_prob[group_unpriv])
        abroca = abs(auc_priv - auc_unpriv)

    # === Fairness score ===
    if not np.isnan(abroca):
        fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3
    else:
        fairness = float("nan")

    # === Print result ===
    print(f"{name:<20} {abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (Privileged: race_African-American == 1)
--------------------------------------------------------------------------
Model                ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
Decision Tree        0.0694     0.0015     -0.1766    0.9175    
Logistic Regression  0.0493     0.0169     -0.2841    0.8832    
Random Forest        0.0252     0.0156     -0.1984    0.9203    
SVM                  0.0537     -0.0010    -0.2368    0.9028    
XGBoost              0.0324     -0.0182    -0.1923    0.9191    


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Load COMPAS dataset ===
df = pd.read_csv("compas_synthetic_data_1000_200_epochs.csv")

# === Extract features and label ===
X = df.drop(columns='two_year_recid')
y = df['two_year_recid']

# === Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Save group membership for fairness analysis
group_priv = X_test['race_African-American'] == 1   # privileged group
group_unpriv = X_test['race_African-American'] == 0 # non-privileged group

# === Define models ===
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Print header ===
print("\nFairness Metrics (Privileged: race_African-American == 1)")
print("--------------------------------------------------------------------------")
print(f"{'Model':<20} {'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")

# === Evaluate each model ===
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # === ERD ===
    err_priv = np.mean(y_pred[group_priv] != y_test[group_priv])
    err_unpriv = np.mean(y_pred[group_unpriv] != y_test[group_unpriv])
    erd = err_unpriv - err_priv

    # === TPRD ===
    def tpr(y_true, y_pred):
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_priv = tpr(y_test[group_priv], y_pred[group_priv])
    tpr_unpriv = tpr(y_test[group_unpriv], y_pred[group_unpriv])
    tprd = tpr_unpriv - tpr_priv

    # === ABROCA ===
    def compute_roc_auc(y_true, y_score):
        fpr, tpr_vals, _ = roc_curve(y_true, y_score)
        return auc(fpr, tpr_vals)

    abroca = np.nan
    if y_prob is not None:
        auc_priv = compute_roc_auc(y_test[group_priv], y_prob[group_priv])
        auc_unpriv = compute_roc_auc(y_test[group_unpriv], y_prob[group_unpriv])
        abroca = abs(auc_priv - auc_unpriv)

    # === Fairness score ===
    if not np.isnan(abroca):
        fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3
    else:
        fairness = float("nan")

    # === Print result ===
    print(f"{name:<20} {abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (Privileged: race_African-American == 1)
--------------------------------------------------------------------------
Model                ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
Decision Tree        0.0042     0.0034     -0.1296    0.9542    
Logistic Regression  0.1202     0.0765     -0.1111    0.8974    
Random Forest        0.0138     -0.0285    0.0556     0.9674    
SVM                  0.0920     0.0660     -0.0741    0.9226    
XGBoost              0.0087     0.0211     -0.0556    0.9716    


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Load COMPAS dataset ===
df = pd.read_csv("generated_data_CLLM_prompt_COMPAS.csv")

# === Extract features and label ===
X = df.drop(columns='two_year_recid')
y = df['two_year_recid']

# === Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Save group membership for fairness analysis
group_priv = X_test['race_African-American'] == 1   # privileged group
group_unpriv = X_test['race_African-American'] == 0 # non-privileged group

# === Define models ===
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Print header ===
print("\nFairness Metrics (Privileged: race_African-American == 1)")
print("--------------------------------------------------------------------------")
print(f"{'Model':<20} {'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")

# === Evaluate each model ===
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # === ERD ===
    err_priv = np.mean(y_pred[group_priv] != y_test[group_priv])
    err_unpriv = np.mean(y_pred[group_unpriv] != y_test[group_unpriv])
    erd = err_unpriv - err_priv

    # === TPRD ===
    def tpr(y_true, y_pred):
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_priv = tpr(y_test[group_priv], y_pred[group_priv])
    tpr_unpriv = tpr(y_test[group_unpriv], y_pred[group_unpriv])
    tprd = tpr_unpriv - tpr_priv

    # === ABROCA ===
    def compute_roc_auc(y_true, y_score):
        fpr, tpr_vals, _ = roc_curve(y_true, y_score)
        return auc(fpr, tpr_vals)

    abroca = np.nan
    if y_prob is not None:
        auc_priv = compute_roc_auc(y_test[group_priv], y_prob[group_priv])
        auc_unpriv = compute_roc_auc(y_test[group_unpriv], y_prob[group_unpriv])
        abroca = abs(auc_priv - auc_unpriv)

    # === Fairness score ===
    if not np.isnan(abroca):
        fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3
    else:
        fairness = float("nan")

    # === Print result ===
    print(f"{name:<20} {abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (Privileged: race_African-American == 1)
--------------------------------------------------------------------------
Model                ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
Decision Tree        0.0272     -0.0401    0.0406     0.9640    
Logistic Regression  0.0018     -0.0110    -0.0298    0.9858    
Random Forest        0.0126     -0.0071    -0.0161    0.9881    
SVM                  0.0075     0.0581     -0.0447    0.9632    
XGBoost              0.0062     0.0221     -0.0447    0.9757    


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Load COMPAS dataset ===
df = pd.read_csv("generated_data_Our_prompt_COMPAS.csv")

# === Extract features and label ===
X = df.drop(columns='two_year_recid')
y = df['two_year_recid']

# === Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Save group membership for fairness analysis
group_priv = X_test['race_African-American'] == 1   # privileged group
group_unpriv = X_test['race_African-American'] == 0 # non-privileged group

# === Define models ===
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Print header ===
print("\nFairness Metrics (Privileged: race_African-American == 1)")
print("--------------------------------------------------------------------------")
print(f"{'Model':<20} {'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")

# === Evaluate each model ===
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # === ERD ===
    err_priv = np.mean(y_pred[group_priv] != y_test[group_priv])
    err_unpriv = np.mean(y_pred[group_unpriv] != y_test[group_unpriv])
    erd = err_unpriv - err_priv

    # === TPRD ===
    def tpr(y_true, y_pred):
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_priv = tpr(y_test[group_priv], y_pred[group_priv])
    tpr_unpriv = tpr(y_test[group_unpriv], y_pred[group_unpriv])
    tprd = tpr_unpriv - tpr_priv

    # === ABROCA ===
    def compute_roc_auc(y_true, y_score):
        fpr, tpr_vals, _ = roc_curve(y_true, y_score)
        return auc(fpr, tpr_vals)

    abroca = np.nan
    if y_prob is not None:
        auc_priv = compute_roc_auc(y_test[group_priv], y_prob[group_priv])
        auc_unpriv = compute_roc_auc(y_test[group_unpriv], y_prob[group_unpriv])
        abroca = abs(auc_priv - auc_unpriv)

    # === Fairness score ===
    if not np.isnan(abroca):
        fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3
    else:
        fairness = float("nan")

    # === Print result ===
    print(f"{name:<20} {abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (Privileged: race_African-American == 1)
--------------------------------------------------------------------------
Model                ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
Decision Tree        0.0349     -0.0131    0.1147     0.9457    
Logistic Regression  0.0399     0.0267     -0.0187    0.9716    
Random Forest        0.0397     0.0472     -0.0142    0.9663    
SVM                  0.0316     0.0201     -0.0705    0.9593    
XGBoost              0.0122     -0.0296    0.0420     0.9721    
